In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, IntegerType, DateType, TimestampType, FloatType

catalog_name = 'ecommerce'

In [0]:
df_bronze_categories = spark.table(f"{catalog_name}.bronze.brz_categories")

In [0]:
#1.1 transformation: slv_categoies -> category_code

df_silver_categories = df_bronze_categories.withColumn("category_code", F.upper(F.col("category_code"))) \
    .dropDuplicates(["category_code"])

In [0]:
#1.2 Validation: slv_categories -> category_code

df_silver_categories.groupBy("category_code") \
    .count() \
    .filter(F.col("count") > 1) \
    .show()

In [0]:
# 2.1 transformation: slv_categories -> category_name

df_silver_categories = df_silver_categories.withColumn("category_name", F.trim(F.initcap(F.col("category_name"))))

In [0]:
# 2.2 Validation: slv_categories -> category_name

df_silver_categories.groupBy("category_name") \
    .count() \
    .filter(F.col("count") > 1) \
    .show()

In [0]:
# 3.1 Quarantine Bad Data -> df_silver_categories_quarantine & df_silver_categories_clean

df_silver_categories_quarantine = df_silver_categories.filter(
    F.col("category_code").isNull() | (F.col("category_code") == "")
) \
    .withColumn("rejection_reason", F.lit("null or empty category_code"))

df_silver_categories_quarantine = df_silver_categories_quarantine.union(
    df_silver_categories.filter(F.col("category_name").isNull() | (F.col("category_name") == "")) \
    .withColumn("rejection_reason", F.lit("null or empty category_name"))
)

df_silver_categories_clean = df_silver_categories.filter(
    F.col("category_code").isNotNull() & (F.col("category_code") != "") &
    F.col("category_name").isNotNull() & (F.col("category_name") != "")
)



In [0]:
# 3.2 Check Quarantined Categories

df_silver_categories_quarantine.show()

In [0]:
# 4.1 Write to Delta: df_silver_categories_quarantine & df_silver_categories_clean

df_silver_categories_quarantine.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog_name}.silver.slv_categories_quarantine")

df_silver_categories_clean.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog_name}.silver.slv_categories_clean")